# Chapter 9: Hash Tables

The programs from Chapter 9 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Separate Chaining

`chained_map.py`

In [1]:
"""A hash map with separate chaining.

Each bucket is a small list of [key, value] pairs. A key's hash picks
its bucket; the bucket is searched with ==. When the average bucket
holds more than 0.75 pairs, the map doubles its buckets and rehashes.
"""


class ChainedMap:
    MAX_LOAD = 0.75

    def __init__(self, buckets=8):
        self._buckets = [[] for _ in range(buckets)]
        self._n = 0

    def __len__(self):
        return self._n

    def _bucket(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def __getitem__(self, key):
        for pair in self._bucket(key):
            if pair[0] == key:
                return pair[1]
        raise KeyError(key)

    def __setitem__(self, key, value):
        bucket = self._bucket(key)
        for pair in bucket:
            if pair[0] == key:
                pair[1] = value  # an existing key: replace its value
                return
        bucket.append([key, value])
        self._n += 1
        if self._n > self.MAX_LOAD * len(self._buckets):
            self._rehash(2 * len(self._buckets))

    def __delitem__(self, key):
        bucket = self._bucket(key)
        for i, pair in enumerate(bucket):
            if pair[0] == key:
                bucket.pop(i)
                self._n -= 1
                return
        raise KeyError(key)

    def __contains__(self, key):
        return any(pair[0] == key for pair in self._bucket(key))

    def __iter__(self):
        for bucket in self._buckets:
            for key, _ in bucket:
                yield key

    def _rehash(self, size):
        """Move every pair into `size` new buckets."""
        pairs = [pair for bucket in self._buckets for pair in bucket]
        self._buckets = [[] for _ in range(size)]
        for key, value in pairs:
            self._bucket(key).append([key, value])

`map_demo.py`

In [2]:
"""Look inside: which bucket each key lands in."""

from chained_map import ChainedMap

m = ChainedMap(buckets=4)
for key in [10, 7, 14]:  # a fourth key would make it resize
    m[key] = f"value {key}"
    print(f"hash({key}) % 4 = {hash(key) % 4}")
for k, bucket in enumerate(m._buckets):
    print("bucket", k, bucket)

hash(10) % 4 = 2
hash(7) % 4 = 3
hash(14) % 4 = 2
bucket 0 []
bucket 1 []
bucket 2 [[10, 'value 10'], [14, 'value 14']]
bucket 3 [[7, 'value 7']]


## Open Addressing

`open_map.py`

In [3]:
"""A hash map with open addressing and linear probing.

All pairs live in one table. A key starts at slot hash(key) % size and,
if that slot is taken by another key, tries the next slot, and the next,
wrapping around. A deleted pair leaves a tombstone, so searches for keys
stored further along don't stop early. The table stays at most two
thirds full, counting tombstones, so there is always an empty slot.
"""

EMPTY = None
TOMBSTONE = object()  # marks a slot whose pair was deleted


class OpenMap:
    def __init__(self, size=8):
        self._slots = [EMPTY] * size  # each slot: EMPTY, TOMBSTONE or pair
        self._n = 0  # live pairs
        self._used = 0  # live pairs plus tombstones
        self.probes = 0  # slots examined, for experiments

    def __len__(self):
        return self._n

    def _find(self, key):
        """(slot index, found?) for key. If it isn't there, the index
        is where it should go: the first tombstone or empty slot seen."""
        size = len(self._slots)
        i = hash(key) % size
        free = None
        while True:
            self.probes += 1
            slot = self._slots[i]
            if slot is EMPTY:
                return (i if free is None else free), False
            if slot is TOMBSTONE:
                if free is None:
                    free = i
            elif slot[0] == key:
                return i, True
            i = (i + 1) % size  # linear probing: try the next slot

    def __getitem__(self, key):
        i, found = self._find(key)
        if not found:
            raise KeyError(key)
        return self._slots[i][1]

    def __setitem__(self, key, value):
        i, found = self._find(key)
        if found:
            self._slots[i] = (key, value)
            return
        if self._slots[i] is EMPTY:
            self._used += 1  # a tombstone reused doesn't add to the count
        self._slots[i] = (key, value)
        self._n += 1
        if self._used * 3 > len(self._slots) * 2:
            self._rehash()

    def __delitem__(self, key):
        i, found = self._find(key)
        if not found:
            raise KeyError(key)
        self._slots[i] = TOMBSTONE
        self._n -= 1

    def __contains__(self, key):
        return self._find(key)[1]

    def __iter__(self):
        for slot in self._slots:
            if slot is not EMPTY and slot is not TOMBSTONE:
                yield slot[0]

    def _rehash(self):
        """A fresh table for the live pairs, dropping the tombstones."""
        pairs = [
            s for s in self._slots if s is not EMPTY and s is not TOMBSTONE
        ]
        self._slots = [EMPTY] * max(8, 4 * len(pairs))
        self._n = self._used = 0
        for key, value in pairs:
            self[key] = value

## How Full Is Too Full?

`probe_lengths.py`

In [4]:
"""How many slots does a search look at, as the table fills up?

Linear probing's average for a successful search is about
(1 + 1 / (1 - a)) / 2, where a is the load: the fraction of slots used.
"""

import random

from open_map import OpenMap

rng = random.Random(2026)
size = 100_000
print(f"{'load':>5} {'measured':>9} {'formula':>8}")
for load in [0.1, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = OpenMap()
    m._slots = [None] * size  # a fixed size, so we control the load
    m._rehash = lambda: None  # and switch resizing off
    keys = [rng.getrandbits(60) for _ in range(int(load * size))]
    for k in keys:
        m[k] = True
    m.probes = 0
    for k in keys:
        m[k]  # look up every key once
    measured = m.probes / len(keys)
    formula = (1 + 1 / (1 - load)) / 2
    print(f"{load:>5} {measured:>9.2f} {formula:>8.2f}")

 load  measured  formula
  0.1      1.06     1.06
  0.3      1.21     1.21
  0.5      1.50     1.50
  0.6      1.77     1.75


  0.7      2.14     2.17
  0.8      2.98     3.00


  0.9      5.34     5.50


## Timing Lookups

`bench_maps.py`

In [5]:
"""Lookups in our two maps and in Python's dict, as they grow."""

import bench
from chained_map import ChainedMap
from open_map import OpenMap


def filled(make):
    def make_input(n):
        m = make()
        for i in range(n):
            m[i * 7919] = i  # spread-out integer keys
        return m, (n // 2) * 7919

    return make_input


def lookup(m, key):
    return m[key]


sizes = [1_000, 10_000, 100_000, 1_000_000]
results = {
    "dict": bench.doubling(lookup, filled(dict), sizes),
    "ChainedMap": bench.doubling(lookup, filled(ChainedMap), sizes),
    "OpenMap": bench.doubling(lookup, filled(OpenMap), sizes),
}
bench.table(results)

         n            dict      ChainedMap         OpenMap
     1,000         79.3 ns          175 ns          224 ns
    10,000    79.2 ns x1.0     164 ns x0.9     232 ns x1.0
   100,000    77.7 ns x1.0     178 ns x1.1     235 ns x1.0
 1,000,000    79.1 ns x1.0     202 ns x1.1     252 ns x1.1


## When Hashing Goes Wrong

`hash_flood.py`

In [6]:
"""A hash flood: keys that all hash to the same value turn a dict's
constant-time operations into linear ones, and filling it quadratic."""

import bench


class Key:
    def __init__(self, x, flood):
        self.x = x
        self.flood = flood

    def __hash__(self):
        return 42 if self.flood else hash(self.x)

    def __eq__(self, other):
        return self.x == other.x


def fill(n, flood):
    d = {}
    for i in range(n):
        d[Key(i, flood)] = i


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "good hashes": bench.doubling(fill, lambda n: (n, False), sizes),
    "all hash to 42": bench.doubling(fill, lambda n: (n, True), sizes),
}
bench.table(results)
bench.chart(results, "ch09_flood.png", "Filling a dict: a hash flood")

         n     good hashes  all hash to 42
       500         59.5 us         5.02 ms
     1,000     117 us x2.0    20.1 ms x4.0
     2,000     261 us x2.2    81.5 ms x4.0
     4,000     533 us x2.0     340 ms x4.2
